# 상태 변경 알림 중복·구문 미반영 진단

## 확인 계획
1. 실행 중인 Docker Compose 백엔드 컨테이너와 이미지 생성 시각을 확인한다.
2. 상태 변경 API가 한 번의 사용자 동작에서 여러 번 호출되는지 로그와 프론트 이벤트 경로를 확인한다.
3. 백엔드 알림 생성 경로와 중복 수신 가능성을 확인한다.
4. 실행 코드가 최신 알림 문구를 포함하는지 확인한다.


## 진단 결과

- [x] 실행 중인 Docker 이미지와 컨테이너 시작 시각 확인
- [x] 동일 상태 변경 API 중복 호출 여부 확인
- [x] 알림 생성 경로와 중복 수신 가능성 확인
- [x] 변경된 알림 문구가 실행 이미지에 반영됐는지 확인

### 근거

- 실행 중인 `app-backend-spring` 이미지는 2026-07-28 10:36(KST)에 생성되어, 이후 수정한 담당자 이름 기반 알림 문구가 포함되지 않았다.
- nginx 접근 로그에서 2026-07-28 11:25:37(KST)에 `PATCH /api/v1/projects/20/tasks/348/position` 요청이 같은 초에 정확히 두 번 전송되고 모두 200 응답한 사실을 확인했다.
- 직전에 체크리스트 항목 497, 498이 연속 변경됐다. `TaskDetailPanel`은 완료 체크 시 전달받은 업무 상태가 `todo`이면 자동으로 진행 중 이동을 호출하므로, 빠른 연속 체크 동안 오래된 `todo` 상태를 두 핸들러가 함께 보고 중복 호출하는 경쟁 조건이 발생한다.
- 백엔드는 요청 한 건당 알림 한 건을 생성한다. 두 요청이 동시에 이전 상태를 읽으면 두 요청 모두 상태 변경으로 판단하여 중복 알림을 저장할 수 있다.
